# Calcula tu inflación — por marca

Este cuaderno calcula la **inflación real por marca** con los datos abiertos de
*Quién es Quién en los Precios* (Profeco), y detecta **reduflación**: cuando el
producto cuesta casi lo mismo pero trae menos contenido.

**No necesitas saber Python ni instalar nada.** Solo ejecuta las celdas en orden.

Para ejecutar una celda: haz clic en ella y presiona el botón ▶ de la izquierda
(o `Shift + Enter`).

---

## Paso 1 — Preparar el cuaderno

Ejecuta la celda de abajo. Tarda unos segundos y descarga el código del análisis.

In [ ]:
#@title Paso 1: preparar (ejecuta esta celda)
!pip install -q pandas
!wget -q -O inflacion_por_marca.py https://raw.githubusercontent.com/carsam68-MonheyB/CALCULA_TU_INFLACI-N/main/inflacion_por_marca.py

import os, glob
os.makedirs("datos", exist_ok=True)
print("Listo. Ya puedes pasar al Paso 2.")

---

## Paso 2 — Subir tus archivos CSV

Tus datos vienen **un CSV por mes**. Para comparar necesitas **dos**: el mismo
mes de dos años distintos (por ejemplo agosto 2024 y agosto 2025). Usar el mismo
mes evita mezclar efectos de temporada.

Ejecuta la celda y aparecerá un botón **"Elegir archivos"**. Selecciona los dos
CSV desde tu computadora.

> Si un archivo pesa mucho, la subida puede tardar varios minutos. Es normal.
> No cierres la pestaña.

In [ ]:
#@title Paso 2: subir los CSV (ejecuta y elige tus archivos)
from google.colab import files
import shutil, os

subidos = files.upload()
for nombre in subidos:
    shutil.move(nombre, os.path.join("datos", nombre))

print("\nArchivos listos en la carpeta datos/:")
for f in sorted(os.listdir("datos")):
    mb = os.path.getsize(os.path.join("datos", f)) / 1e6
    print(f"  {f}  ({mb:,.0f} MB)")

---

## Paso 3 — Decir cuál archivo es cuál

Escribe el **nombre exacto** de cada archivo, tal como aparece en la lista de
arriba. Cópialos y pégalos entre las comillas.

- `ARCHIVO_BASE` = el año **viejo** (el punto de partida)
- `ARCHIVO_ACTUAL` = el año **nuevo** (con el que comparas)

In [ ]:
#@title Paso 3: indicar los archivos
ARCHIVO_BASE   = "QQP_2024_08.csv"  #@param {type:"string"}
ARCHIVO_ACTUAL = "QQP_2025_08.csv"  #@param {type:"string"}

import os
for etiqueta, f in (("BASE", ARCHIVO_BASE), ("ACTUAL", ARCHIVO_ACTUAL)):
    ruta = os.path.join("datos", f)
    print(f"{etiqueta:7} {f:40} {'OK' if os.path.exists(ruta) else '<-- NO EXISTE, revisa el nombre'}")

---

## Paso 4 — Elegir qué analizar

**Importante:** un año de QQP puede traer más de 20 millones de precios. Si no
filtras nada, el cuaderno se queda sin memoria. **Siempre pon al menos un
filtro.**

Escribe los términos separados por espacios. Deja vacío `""` lo que no quieras usar.

Ejemplos:
- `PRODUCTO = "DESODORANTE"` → solo desodorantes
- `PRODUCTO = "DESODORANTE SHAMPOO"` → desodorantes **o** shampoos
- `CATEGORIA = "CUIDADO PERSONAL"` → toda la categoría
- `ESTADO = "COAHUILA"` → solo ese estado

In [ ]:
#@title Paso 4: filtros
PRODUCTO  = "DESODORANTE"  #@param {type:"string"}
MARCA     = ""             #@param {type:"string"}
CATEGORIA = ""             #@param {type:"string"}
ESTADO    = ""             #@param {type:"string"}
CADENA    = ""             #@param {type:"string"}

POR_CADENA = True  #@param {type:"boolean"}
MIN_OBSERVACIONES = 3  #@param {type:"integer"}

if not any((PRODUCTO, MARCA, CATEGORIA, ESTADO, CADENA)):
    print("AVISO: no pusiste ningun filtro. Con archivos grandes esto puede")
    print("       agotar la memoria. Escribe al menos un producto o categoria.")
else:
    print("Filtros listos. Pasa al Paso 5.")

---

## Paso 5 — Calcular

Ejecuta y espera. Con archivos grandes puede tardar varios minutos: verás un
contador de filas mientras avanza.

In [ ]:
#@title Paso 5: calcular (ejecuta esta celda)
import sys, os

cmd = [sys.executable, "inflacion_por_marca.py",
       "--base",   os.path.join("datos", ARCHIVO_BASE),
       "--actual", os.path.join("datos", ARCHIVO_ACTUAL),
       "--min-obs", str(MIN_OBSERVACIONES),
       "--csv", "resultado.csv"]

for bandera, valor in (("--producto", PRODUCTO), ("--marca", MARCA),
                       ("--categoria", CATEGORIA), ("--estado", ESTADO),
                       ("--cadena", CADENA)):
    if valor.strip():
        cmd += [bandera] + valor.split()

if POR_CADENA:
    cmd.append("--por-cadena")

import subprocess
proc = subprocess.run(cmd, capture_output=True, text=True)
print(proc.stdout)
if proc.returncode != 0:
    print("--- detalle ---")
    print(proc.stderr[-3000:])

---

## Paso 6 — Descargar el resultado

Guarda la tabla en tu computadora para abrirla en Excel.

In [ ]:
#@title Paso 6: descargar resultado.csv
from google.colab import files
import os

if os.path.exists("resultado.csv"):
    files.download("resultado.csv")
else:
    print("Todavia no hay resultado.csv. Ejecuta el Paso 5 primero.")

---

## Cómo leer la tabla de reduflación

| Columna | Qué significa |
|---|---|
| **ETIQUETA** | Cuánto subió el precio que ves en el anaquel |
| **CONTENIDO** | Cuánto cambió el tamaño del empaque (negativo = encogió) |
| **REAL** | Cuánto subió el precio por gramo o mililitro |
| **BRECHA** | REAL menos ETIQUETA — la inflación que no se ve |

Una marca con `<-- ENCOGIO` bajó el contenido del empaque. Si su BRECHA es
grande, estás pagando bastante más por gramo aunque el precio del anaquel
casi no se haya movido.

## Notas

- Se usa la **mediana** de precios, no el promedio, para que unos pocos
  registros mal capturados no distorsionen el resultado.
- Un artículo solo aparece si está en **ambos** periodos con al menos
  `MIN_OBSERVACIONES` registros.
- Los archivos que subes se borran cuando cierras la sesión de Colab. Descarga
  siempre tu `resultado.csv`.

Código y documentación: <https://github.com/carsam68-MonheyB/CALCULA_TU_INFLACI-N>